In [ ]:
import numpy as np
import os
import mne
import pandas as pd
import utils
import utils_stream
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import copy
from itertools import product
from numpy import linalg as LA
from scipy.io import loadmat
from scipy.linalg import block_diag
from scipy.stats import wilcoxon
from cca_zoo.linear import MCCA, rCCA
from algo_suppl import CorrCA_LW, MCCA_LW
from algo_ccazoo import CorrelationAnalysis
%matplotlib widget

In [ ]:
# # Results aggregation (Neetha Dataset)
# traintrials = [5, 10, 15, 20, 25, 30, 35, 40 ,45]
# mean_all_nbtrials = {}
# std_all_trials = {}
# for traintrial in traintrials:
#     update_len = 60
#     SHUFFLE = True
#     SEEDS = [16, 32, 64, 128]
#     res_all_dict = {}
#     for SEED in SEEDS:
#         file_path = f"tables/Neetha/iter/nbtraintrials{traintrial}_nbdisconnected0_updatelen{update_len}_predlen30_hparadata4_3_hparafeats6_0_evalpara3_2_SHUFFLETrue_SEED{SEED}.pkl"
#         with open(file_path, 'rb') as f:
#             res_dict = pickle.load(f)
#         for key in res_dict.keys():
#             if key not in res_all_dict:
#                 res_all_dict[key] = []
#             res_all_dict[key].append(res_dict[key])
#     for key in res_all_dict.keys():
#         res_all_dict[key] = np.concatenate(res_all_dict[key], axis=0)
#         if key == 'cpu_time_single':
#             baseline = res_all_dict[key]
#         if key.startswith('cpu_time'):
#             res_all_dict[key] = res_all_dict[key] / baseline
#         res_avg = np.mean(res_all_dict[key], axis=0)
#         res_std = np.std(res_all_dict[key], axis=0)
#         # check if type is np.ndarray
#         if isinstance(res_avg, np.ndarray):
#             res_avg = res_avg[-1]
#             res_std = res_std[-1]
#         if key not in mean_all_nbtrials:
#             mean_all_nbtrials[key] = []
#             std_all_trials[key] = []
#         mean_all_nbtrials[key].append(res_avg)
#         std_all_trials[key].append(res_std)

In [ ]:
# Results aggregation (Fuglsang2018 Dataset)
traintrials = [6, 12, 18, 24, 30, 36, 42]
mean_all_nbtrials = {}
std_all_trials = {}
for traintrial in traintrials:
    update_len = 50
    SHUFFLE = True
    SEEDS = [16, 32, 64, 128]
    res_all_dict = {}
    for SEED in SEEDS:
        file_path = f"tables/fuglsang2018/iter/nbtraintrials{traintrial}_nbdisconnected0_updatelen{update_len}_predlen25_hparadata4_3_hparafeats9_0_evalpara3_2_SHUFFLETrue_SEED{SEED}.pkl"
        with open(file_path, 'rb') as f:
            res_dict = pickle.load(f)
        for key in res_dict.keys():
            if key not in res_all_dict:
                res_all_dict[key] = []
            res_all_dict[key].append(res_dict[key])
    for key in res_all_dict.keys():
        res_all_dict[key] = np.concatenate(res_all_dict[key], axis=0)
        if key == 'cpu_time_single':
            baseline = res_all_dict[key]
        if key.startswith('cpu_time'):
            res_all_dict[key] = res_all_dict[key] / baseline
        res_avg = np.mean(res_all_dict[key], axis=0)
        res_std = np.std(res_all_dict[key], axis=0)
        # check if type is np.ndarray
        if isinstance(res_avg, np.ndarray):
            res_avg = res_avg[-1]
            res_std = res_std[-1]
        if key not in mean_all_nbtrials:
            mean_all_nbtrials[key] = []
            std_all_trials[key] = []
        mean_all_nbtrials[key].append(res_avg)
        std_all_trials[key].append(res_std)

In [ ]:
def create_results_dataframe(mean_all_nbtrials, std_all_trials, traintrials):
    """Convert your existing data structure to a clean DataFrame"""

    key_to_method = {
        'acc_sup': 'Supervised',
        'acc_single': 'Single-Enc',
        'acc_single_unbiased': 'Unbiased',
        'acc_two': 'Two-Enc',
        'acc_bpsk_local': 'Soft',
        'acc_single_warminit': 'SumInit',
        'acctrain_single': 'Single-Enc',
        'acctrain_single_unbiased': 'Unbiased',
        'acctrain_two': 'Two-Enc',
        'acctrain_bpsk_local': 'Soft',
        'acctrain_single_warminit': 'SumInit',
        'cpu_time_single': 'Single-Enc',
        'cpu_time_single_unbiased': 'Unbiased',
        'cpu_time_two': 'Two-Enc',
        'cpu_time_bpsk_local': 'Soft',
        'cpu_time_single_warminit': 'SumInit'}

    data_rows = []

    for key in mean_all_nbtrials.keys():
        if key.startswith('cpu_time'):
            TYPE = 'CPU Time Overhead'
        elif key.startswith('acc_'):
            TYPE = 'Inductive Accuracy'
        else:
            TYPE = 'Transductive Accuracy'
        for i, train_size in enumerate(traintrials):
            data_rows.append({
                'training_set_size': train_size,
                'method': key_to_method[key],
                'mean': mean_all_nbtrials[key][i],  # Convert to percentage
                'std': std_all_trials[key][i] ,
                'data_type': TYPE
            })

    return pd.DataFrame(data_rows)

In [ ]:
# Define consistent styling
method_colors = {
    'Supervised': '#1f77b4',      # Blue
    'Single-Enc': '#ff7f0e',      # Orange  
    'Unbiased': '#2ca02c',          # Green (assuming this is your method name)
    'Two-Enc': '#d62728',         # Red
    'Soft': '#9467bd',            # Purple
    'SumInit': '#8c564b'          # Brown
}

method_markers = {
    'Supervised': 'o',
    'Single-Enc': 's', 
    'Unbiased': '^',
    'Two-Enc': 'D',
    'Soft': 'v',
    'SumInit': 'p'
}

method_linestyles = {
    'Supervised': '-',      # Solid line
    'Single-Enc': '-',     # Dashed line
    'Unbiased': '-',         # Dash-dot line
    'Two-Enc': ':',         # Dotted line
    'Soft': '-.',            # Solid line
    'SumInit': '--'         # Dashed line
}

In [ ]:
def plot_grouped_results(df, data_type, ax, show_legend=True, show_xlabel=True, logy=False, logx=False, trial_len=60):
    training_sizes = sorted(df['training_set_size'].unique())
    methods = df['method'].unique()
    n_methods = len(methods)
    n_sizes = len(training_sizes)

    # Calculate bar positions
    bar_width = 0.8 / n_methods
    x_positions = np.arange(n_sizes)

    handles = []
    labels = []

    for i, method in enumerate(methods):
        method_data = df[df['method'] == method]
        means, stds, group_positions = [], [], []

        for j, size in enumerate(training_sizes):
            size_data = method_data[method_data['training_set_size'] == size]
            if len(size_data) > 0:
                means.append(size_data['mean'].iloc[0])
                stds.append(size_data['std'].iloc[0])
            else:
                means.append(np.nan)
                stds.append(np.nan)

            pos = x_positions[j] + (i - (n_methods - 1) / 2) * bar_width
            group_positions.append(pos)

        if any(~np.isnan(means)):
            line = ax.plot(
                group_positions, means,
                label=method,
                marker=method_markers.get(method, 'o'),
                linestyle=method_linestyles.get(method, '-'),
                color=method_colors.get(method, f'C{i}'),
                linewidth=2.5,
                markersize=8,
                alpha=0.85,
                markeredgewidth=1.5,
                markeredgecolor='white'
            )[0]

            handles.append(line)
            labels.append(method)

            ax.errorbar(
                group_positions, means, yerr=stds,
                fmt='none',
                ecolor=method_colors.get(method, f'C{i}'),
                capsize=4, capthick=1.5,
                elinewidth=1.5,
                alpha=0.4
            )

    # Labels and ticks
    if show_xlabel:
        ax.set_xlabel('Training Set Size (min)', fontsize=12)
        ax.set_xticks(x_positions)
        ax.set_xticklabels([int(size/60*trial_len) for size in training_sizes], fontsize=10)
    else:
        ax.set_xticks(x_positions)
        ax.set_xticklabels([])

    if data_type != 'CPU Time Overhead':
        ax.set_ylabel(data_type, fontsize=12)
    else:
        ax.set_ylabel('Normalized CPU Time', fontsize=12)

    # Apply logarithmic scaling if requested
    if logy:
        ax.set_yscale('log')
    if logx:
        ax.set_xscale('log')

    if show_legend:
        ax.legend(fontsize=10, loc='best')

    ax.grid(True, alpha=0.3, which='both', axis='y')

    for i in range(1, len(training_sizes)):
        ax.axvline(x=i - 0.5, color='gray', linestyle='--', alpha=0.3)

    return handles, labels


In [ ]:
import matplotlib.gridspec as gridspec
plt.close('all')
fig = plt.figure(figsize=(7, 8))
gs = gridspec.GridSpec(3, 1, height_ratios=[3, 3, 2], hspace=0.1)
axes = [fig.add_subplot(gs[i]) for i in range(3)]

df = create_results_dataframe(mean_all_nbtrials, std_all_trials, traintrials)

# Plot all subplots without individual legends
for i, data_type in enumerate(['Transductive Accuracy', 'Inductive Accuracy', 'CPU Time Overhead']):
    show_xlabel = (i == 2)
    logy = (data_type == 'CPU Time Overhead')
    h, l = plot_grouped_results(df[df['data_type'] == data_type], data_type, axes[i], show_legend=False, show_xlabel=show_xlabel, logy=logy, trial_len=update_len)
    if i == 1:  # Store handles from first subplot
        handles, labels = h, l

# Create single legend at top of figure
labels = [l if l != 'Single-Enc' else 'Single-Enc (Baseline)' for l in labels]
labels = [l if l != 'Unbiased' else 'Cross-Validated [8]' for l in labels]
order = [0, 3, 1, 4, 5, 2]
handles = [handles[i] for i in order]
labels = [labels[i] for i in order]
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=3, fontsize=10)

plt.tight_layout()
plt.subplots_adjust(top=0.88)  # Make room for legend
plt.show()
plt.savefig('res_fuglsang2018.pdf', bbox_inches='tight', dpi=600, pad_inches=0.01)